# Part 4: Complete RAG Pipeline

In this notebook, we'll:
1. Build the complete RAG (Retrieval-Augmented Generation) pipeline
2. Load documents and create embeddings
3. Retrieve relevant documents
4. Generate responses using an LLM
5. Test with various queries

## Import and Initialize

In [ ]:
import sys
sys.path.append('..')

from src.rag_chain import RAGPipeline
from src.pdf_loader import PDFLoader
from src.utils import ConsoleFormatter, print_results
from dotenv import load_dotenv
import json

# Load environment variables
load_dotenv('..')

print("✓ Libraries imported")

## Initialize RAG Pipeline

In [ ]:
# Create RAG pipeline with local embeddings
# For OpenAI, change embeddings_type and llm_type to "openai"
rag = RAGPipeline(
    embeddings_type="local",  # Free, fast, runs locally
    llm_type="openai",  # Uses OpenAI or fake LLM
    model_name="gpt-3.5-turbo",
    temperature=0.7
)

print("✓ RAG pipeline initialized")

# Display pipeline info
info = rag.get_pipeline_info()
print("\nPipeline Configuration:")
print(json.dumps(info, indent=2, default=str))

## Load Documents

In [ ]:
# Load documents from data directory
# This will:
# 1. Load PDFs from data/
# 2. Create embeddings
# 3. Store in vector database

print("Loading documents...")
rag.load_documents(pdf_path="../data")
print("✓ Documents loaded and indexed")

# Check vector store status
info = rag.get_pipeline_info()
print(f"\nDocuments in store: {info['vector_store']['document_count']}")

## Query the RAG System

In [ ]:
# Ask a question
question = "What is machine learning?"

print(f"Question: {question}")
print("-" * 60)

# Get answer from RAG pipeline
result = rag.query(question, k=3)

print(f"\nAnswer:\n{result['answer']}")

print(f"\n\nSources Used ({len(result['sources'])}):")
for i, source in enumerate(result['sources'], 1):
    print(f"\n{i}. Document: {source['metadata'].get('source', 'Unknown')}")
    print(f"   Similarity: {source['similarity']:.4f}")
    print(f"   Text: {source['text'][:80]}...")

## Multiple Queries

In [ ]:
# Test with multiple queries
queries = [
    "What is deep learning?",
    "How do neural networks work?",
    "What is artificial intelligence?"
]

results = []

for query in queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    
    result = rag.query(query, k=2)
    
    print(f"\nAnswer:\n{result['answer'][:300]}...")
    print(f"\nSources: {len(result['sources'])} documents retrieved")
    
    results.append({
        'question': query,
        'answer': result['answer'],
        'sources': len(result['sources'])
    })

## Explore Retrieval Quality

In [ ]:
# Analyze retrieval quality with different K values
question = "What is learning in machine learning?"

print(f"Question: {question}")
print()

for k in [1, 3, 5]:
    result = rag.query(question, k=k)
    
    # Calculate average similarity
    avg_similarity = sum(s['similarity'] for s in result['sources']) / len(result['sources']) if result['sources'] else 0
    
    print(f"K={k}:")
    print(f"  Documents retrieved: {len(result['sources'])}")
    print(f"  Average similarity: {avg_similarity:.4f}")
    
    if result['sources']:
        print(f"  Best match similarity: {max(s['similarity'] for s in result['sources']):.4f}")
    print()

## RAG vs Pure LLM Comparison

In [ ]:
# This demonstrates why RAG is useful
print("RAG Benefits:\n")

query = "What is neural network backpropagation?"
result = rag.query(query, k=3)

print(f"Question: {query}")
print()
print("With RAG:")
print(f"  - Retrieved {len(result['sources'])} relevant documents")
print(f"  - Answer is grounded in your knowledge base")
print(f"  - Can verify answer with sources")
print(f"  - Works with documents not in LLM training")
print()
print("Without RAG:")
print(f"  - LLM would answer from training data")
print(f"  - No knowledge of your specific documents")
print(f"  - Risk of hallucination")
print(f"  - Can't cite sources")

## Interactive Query

In [ ]:
# Try your own question!
# Change this to ask your own question

my_question = "Explain supervised learning"

result = rag.query(my_question, k=3)

print(ConsoleFormatter.header("RAG Query Result"))
print(f"Question: {my_question}\n")
print("Answer:")
print(result['answer'])

print("\n" + ConsoleFormatter.subheader("Retrieved Sources"))
for i, source in enumerate(result['sources'], 1):
    print(f"{i}. [Similarity: {source['similarity']:.4f}]")
    print(f"   Source: {source['metadata'].get('source', 'Unknown')}")
    print(f"   Text: {source['text'][:100]}...\n")

## Summary and Next Steps

In [ ]:
print("""
✓ RAG Pipeline Complete!

You have successfully:
  1. Loaded PDF documents
  2. Created vector embeddings
  3. Stored in vector database
  4. Performed semantic search
  5. Generated answers with LLM

Next Steps:
  □ Add your own PDFs to data/ folder
  □ Customize the RAG pipeline for your needs
  □ Fine-tune retrieval parameters (K, threshold)
  □ Optimize prompts for better answers
  □ Deploy as a web application

Customization Ideas:
  - Use OpenAI embeddings for better quality
  - Implement custom prompt templates
  - Add metadata filtering
  - Implement conversation memory
  - Add citation formatting
""")